## Encoding the sequence of text

First train the tokenizer with this notebook [2_BytePairEncoding](./2_BytePairEncoding.ipynb) and save it to disk.

In [1]:
import sys
sys.path.append('..')

Load the new tokenizer.

In [2]:
from minbpe import RegexTokenizer

tokenizer = RegexTokenizer()
tokenizer.load(model_file="../output/tokenizer/fineweb_tokenizer.model")

Encode the data in batches otherwise you will get OOM (Out Of Memory) error. Experiment with the `BATCH_LINES` value until you find the want that uses the most of your RAM without crashing VSCode.

In [3]:
from itertools import islice
from tqdm import tqdm
import subprocess

file_path = "../data/fineweb2_raw.txt"
BATCH_LINES = 100_000
encoded_text_sequence = []

num_lines = int(subprocess.check_output(["wc", "-l", file_path]).split()[0])
num_batches = (num_lines + BATCH_LINES - 1) // BATCH_LINES  # ceiling division

with open(file_path, "r") as f:
    with tqdm(total=num_batches, unit="batch", desc="Tokenizing") as pbar:
        while True:
            lines = list(islice(f, BATCH_LINES))
            if not lines:
                break
            encoded_text_sequence.extend(tokenizer.encode("".join(lines)))
            pbar.update(1)
            pbar.set_postfix(tokens=f"{len(encoded_text_sequence):,}")

print(f"Total tokens: {len(encoded_text_sequence):,}")

Tokenizing: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 50/50 [9:11:24<00:00, 661.69s/batch, tokens=2,463,713,896]

Total tokens: 2,463,713,896


Save the encoded data so that we can load it later.

In [4]:
import numpy as np

output_path = "../output/encoded_data/encoded_fineweb.npy"
np.save(output_path, np.array(encoded_text_sequence, dtype=np.int64))

# Free up memory
del encoded_text_sequence